In [1]:
from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:8080/v1",
    api_key="not-needed"
)

response = client.chat.completions.create(
    model="Bonsai-27B-Q1_0",
    messages=[
        {
            "role": "user",
            "content": "Say hello in one sentence."
        }
    ],
    temperature=0.7
)

print(response.choices[0].message.content)

Hello!


In [2]:
from optimum.intel.openvino import OVModelForVisualCausalLM
from transformers import AutoProcessor
from PIL import Image
import requests

model_id = "OpenVINO/gemma-4-E2B-it-int8-ov"
processor = AutoProcessor.from_pretrained(model_id)
model = OVModelForVisualCausalLM.from_pretrained(model_id)

url = "https://github.com/openvinotoolkit/openvino_notebooks/assets/29454499/d5fbbd1a-d484-415c-88cb-9986625b7b11"
image = Image.open(requests.get(url, stream=True).raw)

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": "What is unusual in this picture?"},
        ],
    }
]

text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = processor(text=text, images=[image], return_tensors="pt")
input_len = inputs["input_ids"].shape[-1]

output = model.generate(**inputs, do_sample=False, max_new_tokens=100)
response = processor.decode(output[0][input_len:], skip_special_tokens=True)
print(response)


c:\Users\cyril.leconte\Code\Simplon\horragor_brief3\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\cyril.leconte\Code\Simplon\horragor_brief3\.venv\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\cyril.leconte\.cache\huggingface\hub\models--OpenVINO--gemma-4-E2B-it-int8-ov. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode o

KeyboardInterrupt: 

In [8]:
from openvino_genai import LLMPipeline

pipe = LLMPipeline(
    r"./models/OpenVINO/Qwen3-8B-int4-cw-ov",
    device="NPU"
)

print(pipe.generate("Hello"))


RuntimeError: Exception from C:\Jenkins\workspace\private-ci\ie\build-windows-vs2022@3\b\repos\openvino.genai\src\cpp\src\utils.cpp:454:
Could not find a model in the directory '"./models/OpenVINO/Qwen3-8B-int4-cw-ov"'


In [ ]:
hf download OpenVINO/gemma-4-E2B-it-int8-ov --local-dir models/gemma-4-E2B-it-int8-ov